In [ ]:
import pandas as pd

In [ ]:
df = pd.read_stata('/content/BDKR81FL.DTA')

In [ ]:
df.head()

,caseid,bidx,v000,v001,v002,v003,v004,v005,v006,v007,...,s446,idx96,s612i,s612k,s612w,s626i,s626k,s626w,s631j,s631k
0,1 21 2,1,BD8,1,21,2,1,156207,9,2022,...,all of the time,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 29 2,1,BD8,1,29,2,1,156207,9,2022,...,all of the time,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 43 3,1,BD8,1,43,3,1,156207,9,2022,...,NaN,1.0,NaN,NaN,NaN,no,no,no,no,no
3,1 63 3,1,BD8,1,63,3,1,156207,9,2022,...,all of the time,1.0,no,no,no,no,no,no,no,no
4,1 85 2,1,BD8,1,85,2,1,156207,9,2022,...,NaN,1.0,NaN,NaN,NaN,no,no,no,no,no


In [ ]:
selected_columns = ('caseid','h11','h13','v106','v190','v024','v025','b4','hw1')

# Identify columns that are present in the DataFrame
existing_columns = [col for col in selected_columns if col in df.columns]

# Identify columns that are not present in the DataFrame
missing_columns = [col for col in selected_columns if col not in df.columns]

if missing_columns:
    print(f"Warning: The following columns were not found in the DataFrame: {missing_columns}")

if existing_columns:
    analysis_df = df[existing_columns].copy()
    print("Original dataset shape:",df.shape)
    print("Filtered dataset shape:",analysis_df.shape)
else:
    print("Error: No valid columns from the selection were found in the DataFrame to create analysis_df.")
    analysis_df = pd.DataFrame() # Create an empty DataFrame if no valid columns are found

Original dataset shape: (8784, 1238)
Filtered dataset shape: (8784, 9)


In [ ]:
analysis_df.head()

,caseid,h11,h13,v106,v190,v024,v025,b4,hw1
0,1 21 2,no,NaN,higher,richer,barishal,urban,female,NaN
1,1 29 2,no,NaN,secondary,richer,barishal,urban,female,NaN
2,1 43 3,no,NaN,secondary,richest,barishal,urban,male,46.0
3,1 63 3,"yes, last two weeks",yes: ors - probed,higher,richest,barishal,urban,female,NaN
4,1 85 2,no,NaN,secondary,middle,barishal,urban,male,46.0


In [ ]:
diarrhea_df = analysis_df[analysis_df['h11'].isin(['yes',1])].copy()
cleaned_df=diarrhea_df.dropna(subset=['v106','v190','h13'])
print("Total children with diarrhea in the clean dataset:",cleaned_df.shape[0])
print("/ndistribution of ORS treatment (target-h13):")
print(cleaned_df['h13'].value_counts())


Total children with diarrhea in the clean dataset: 0
/ndistribution of ORS treatment (target-h13):
h13
no                   0
yes: ors - probed    0
Name: count, dtype: int64


In [ ]:
print(analysis_df['h11'].value_counts(dropna=False))


h11
no                     8075
yes, last two weeks     410
NaN                     276
don't know               23
Name: count, dtype: int64


In [ ]:
diarrhea_df = analysis_df[analysis_df['h11'] == 'yes, last two weeks'].copy()
cleaned_df = diarrhea_df.dropna(subset=['v106','v190','h13'])
print("Total children with diarrhea in the clean dataset:",cleaned_df.shape[0])
print("/ndistribution of ORS treatment (Target-h13):")
print(cleaned_df['h13'].value_counts())

Total children with diarrhea in the clean dataset: 410
/ndistribution of ORS treatment (Target-h13):
h13
yes: ors - probed    303
no                   107
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np # Added numpy import
from sklearn.model_selection import train_test_split
# from re import X # This import is not used and shadows the variable X. Removed.

required_cols = ['h11', 'v106', 'v190', 'v024', 'v025', 'b4', 'hw1']
full_analysis_df = df[required_cols].copy()

def clean_target(val):
  val_str = str(val).lower().strip()
  if 'yes' in val_str:
    return 1  # Changed from 0 to 1 for positive case (Diarrhea)
  elif 'no' in val_str: # Added to handle 'no' cases as 0
    return 0
  else:
    return np.nan

full_analysis_df['target'] = full_analysis_df['h11'].apply(clean_target)
full_analysis_df = full_analysis_df.dropna(subset=['target'])

# Define X and y
# Features for X: 'v106', 'v190', 'v024', 'v025', 'b4', 'hw1'
# Target for y: 'target'
feature_columns_for_X = ['v106', 'v190', 'v024', 'v025', 'b4', 'hw1']
X = full_analysis_df[feature_columns_for_X]
y = full_analysis_df['target']

# Separate categorical and numerical features for X preprocessing
categorical_cols_for_dummies = ['v106', 'v190', 'v024', 'v025', 'b4']
numerical_cols_for_X = ['hw1'] # Assuming hw1 is a numerical feature

X_categorical = X[categorical_cols_for_dummies]
X_numerical = X[numerical_cols_for_X]

X_encoded_categorical = pd.get_dummies(X_categorical, columns=categorical_cols_for_dummies, drop_first=True)
X_encoded = pd.concat([X_encoded_categorical, X_numerical], axis=1)

# Ensure 'target' is converted to integer type (0 or 1)
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
print("--- Data Preparation Complete ---")
print(f"Total dataset size for analysis: {full_analysis_df.shape[0]} rows")
print(f"Diarrhea Cases (Positive - 1): {y.sum()}")
print(f"Non-Diarrhea Cases (Negative - 0): {len(y) - y.sum()}")

--- Data Preparation Complete ---
Total dataset size for analysis: 8508 rows
Diarrhea Cases (Positive - 1): 410
Non-Diarrhea Cases (Negative - 0): 8098


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer # Import SimpleImputer

# Identify the numerical column with NaNs, which is 'hw1'
numerical_cols_for_imputation = ['hw1']

# Initialize the imputer to replace NaNs with the mean of the column
imputer = SimpleImputer(strategy='mean')

# Fit the imputer on the training data and transform both training and testing data
X_train[numerical_cols_for_imputation] = imputer.fit_transform(X_train[numerical_cols_for_imputation])
X_test[numerical_cols_for_imputation] = imputer.transform(X_test[numerical_cols_for_imputation])

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"New Model Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

New Model Accuracy: 58.28%

Confusion Matrix:
[[947 673]
 [ 37  45]]

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.58      0.73      1620
           1       0.06      0.55      0.11        82

    accuracy                           0.58      1702
   macro avg       0.51      0.57      0.42      1702
weighted avg       0.92      0.58      0.70      1702



In [ ]:
import pandas as pd
import numpy as np
coefficients = model.coef_[0]
feature_names = X_train.columns
odds_ratios = np.exp(coefficients)
importance_df = pd.DataFrame({
    'Feature / Risk Factor': feature_names,
    'Coefficient (Beta)': coefficients,
    'Odds Ratio (OR)': odds_ratios
}).sort_values(by='Coefficient (Beta)', ascending=False)
print("--- Socio-demographic Risk Factors for Diarrhea ---")
print(importance_df)

--- Socio-demographic Risk Factors for Diarrhea ---
   Feature / Risk Factor  Coefficient (Beta)  Odds Ratio (OR)
11         v024_rajshahi            0.291937         1.339018
10       v024_mymensingh            0.148293         1.159853
9            v024_khulna            0.109125         1.115302
7        v024_chattogram            0.096434         1.101237
1         v106_secondary            0.058806         1.060570
0           v106_primary           -0.006236         0.993784
16                   hw1           -0.021614         0.978618
3            v190_poorer           -0.068616         0.933685
2            v106_higher           -0.087703         0.916033
15             b4_female           -0.108360         0.897305
14            v025_rural           -0.153104         0.858040
6           v190_richest           -0.253980         0.775707
12          v024_rangpur           -0.376502         0.686258
13           v024_sylhet           -0.393831         0.674468
8             v024